# Verify datasets properties
In this notebook I want to check basic dataset properties generated by us and verify it against a dataset created by main paper authors.

The link to our dataset is: https://huggingface.co/datasets/tkwiecinski/longfact-test-split

The link to original dataset: https://huggingface.co/datasets/obalcells/longfact-augmented-annotations/viewer?views%5B%5D=llama_33_70b_instruct_train

## Diagnostic plan for dataset validity (focus: Apertus split)

This notebook section adds checks that can explain why the probe fails specifically on the Apertus split, before touching model/probe code.

We focus on:
1. **Schema validity**: required fields, annotation format consistency, label vocabulary consistency.
2. **Span validity**: whether each annotated span can be located reliably in the completion at the recorded index.
3. **Label quality signals**: frequency of `Supported` / `Not Supported` / `N/A` / unknown labels.
4. **Distribution shift**: compare Apertus vs Llama split statistics (lengths, span counts, positive rates).
5. **Split integrity**: prompt overlap/leakage between train and test, and cross-subset overlap.
6. **Failure exemplars**: print concrete problematic rows for manual inspection.

These checks are aligned with concerns discussed around LongFact-style annotation quality (span minimality, label noise, long-tail spans).

In [29]:
import hashlib
import numpy as np
import pandas as pd

from datasets import get_dataset_config_names, load_dataset

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [30]:
OUR_REPO = "tkwiecinski/longfact-test-split"
ORIG_REPO = "obalcells/longfact-augmented-annotations"

OUR_SUBSETS = [
    "Apertus_8B_Instruct_2509",
    "Meta_Llama_3.1_8B_Instruct",
]

ORIG_SUBSETS = [
    "Mistral-Small-24B-Instruct-2501",
    "Qwen2.5-7B-Instruct",
    "gemma-2-9b-it",
    "Llama-3.3-70B-Instruct",
    "Meta-Llama-3.1-8B-Instruct"
]   


LABEL_MAP = {
    "Not Supported": 1.0,
    "NS": 1.0,
    "Insufficient Information": 1.0,
    "Supported": 0.0,
    "S": 0.0,
    "N/A": -100.0,
    None: -100.0,
}

def safe_configs(repo: str) -> list[str]:
    try:
        configs = get_dataset_config_names(repo)
        print(f"{repo}: {len(configs)} configs")
        return configs
    except Exception as exc:
        print(f"Could not list configs for {repo}: {exc}")
        return []

def safe_load(repo: str, subset: str, split: str):
    try:
        ds = load_dataset(repo, subset, split=split)
        print(f"Loaded {repo} | {subset} | {split}: {len(ds)} rows")
        return ds
    except Exception as exc:
        print(f"Failed to load {repo} | {subset} | {split}: {exc}")
        return None

def pick_config(configs: list[str], include: str | None = None) -> str | None:
    if not configs:
        return None
    if include is None:
        return configs[0]
    include_l = include.lower()
    return next((cfg for cfg in configs if include_l in cfg.lower()), configs[0])

def _prompt_hash(prompt: str | None) -> str | None:
    if not isinstance(prompt, str):
        return None
    return hashlib.md5(prompt.strip().encode("utf-8")).hexdigest()

def parse_longfact_row(row: dict) -> dict:
    conversation = row.get("conversation")
    prompt = None
    completion = None

    if isinstance(conversation, list) and len(conversation) >= 2:
        user_msg = conversation[-2]
        assistant_msg = conversation[-1]
        if isinstance(user_msg, dict):
            prompt = user_msg.get("content")
        if isinstance(assistant_msg, dict):
            completion = assistant_msg.get("content")

    if completion is None:
        completion = row.get("completion")
    if prompt is None and isinstance(conversation, list) and conversation and isinstance(conversation[0], dict):
        prompt = conversation[0].get("content")

    if row.get("annotations") is not None:
        raw_annotations = row["annotations"]
        source = "annotations"
    elif row.get("verified_entities") is not None:
        raw_annotations = row["verified_entities"]
        source = "verified_entities"
    else:
        raw_annotations = []
        source = "none"

    annotations = []
    for entity in raw_annotations if isinstance(raw_annotations, list) else []:
        if not isinstance(entity, dict):
            continue

        if source == "annotations":
            span = entity.get("span")
            index = entity.get("index")
        elif source == "verified_entities":
            span = entity.get("text")
            index = entity.get("idx")
        else:
            span = entity.get("span") or entity.get("text")
            index = entity.get("index", entity.get("idx"))

        label = entity.get("label")
        annotations.append(
            {
                "span": span,
                "index": index,
                "label": label,
                "label_scalar": LABEL_MAP.get(label),
            }
        )

    return {
        "prompt": prompt,
        "completion": completion,
        "prompt_hash": _prompt_hash(prompt),
        "annotation_source": source,
        "annotations": annotations,
    }

In [31]:
def analyze_longfact_dataset(ds, repo: str, subset: str, split: str):
    rows, spans, issues = [], [], []

    for row_id, row in enumerate(ds):
        parsed = parse_longfact_row(row)
        prompt = parsed["prompt"]
        completion = parsed["completion"]
        annotations = parsed["annotations"]

        row_valid = 0
        row_invalid = 0

        for ann_id, ann in enumerate(annotations):
            span = ann["span"]
            index = ann["index"]
            label = ann["label"]
            label_scalar = ann["label_scalar"]

            issue = None
            matched_at_index = False
            found_anywhere = False

            if not isinstance(span, str) or not span:
                issue = "empty_or_nonstring_span"
            elif not isinstance(completion, str):
                issue = "completion_missing"
            elif isinstance(index, int):
                if index < 0 or index >= len(completion):
                    issue = "index_out_of_bounds"
                elif completion[index : index + len(span)] == span:
                    matched_at_index = True
                elif span in completion:
                    found_anywhere = True
                    issue = "span_not_matching_declared_index_but_found_elsewhere"
                else:
                    issue = "span_not_found_in_completion"
            else:
                if span in completion:
                    found_anywhere = True
                    issue = "missing_or_nonint_index"
                else:
                    issue = "missing_or_nonint_index_and_span_not_found"

            if label_scalar is None and label not in LABEL_MAP:
                issue = "unknown_label" if issue is None else f"{issue}|unknown_label"

            if issue is None:
                row_valid += 1
            else:
                row_invalid += 1
                issues.append(
                    {
                        "repo": repo,
                        "subset": subset,
                        "split": split,
                        "row_id": row_id,
                        "ann_id": ann_id,
                        "issue": issue,
                        "label": label,
                        "index": index,
                        "span_preview": (span[:120] + "...") if isinstance(span, str) and len(span) > 120 else span,
                    }
                )

            spans.append(
                {
                    "repo": repo,
                    "subset": subset,
                    "split": split,
                    "row_id": row_id,
                    "ann_id": ann_id,
                    "label_raw": label,
                    "label_scalar": label_scalar if label_scalar is not None else np.nan,
                    "index": index if isinstance(index, int) else np.nan,
                    "span_len_chars": len(span) if isinstance(span, str) else np.nan,
                    "matched_at_index": matched_at_index,
                    "found_anywhere": found_anywhere,
                    "is_valid": issue is None,
                }
            )

        rows.append(
            {
                "repo": repo,
                "subset": subset,
                "split": split,
                "row_id": row_id,
                "prompt_hash": parsed["prompt_hash"],
                "prompt_len": len(prompt) if isinstance(prompt, str) else np.nan,
                "completion_len": len(completion) if isinstance(completion, str) else np.nan,
                "annotation_source": parsed["annotation_source"],
                "n_annotations": len(annotations),
                "n_valid_spans": row_valid,
                "n_invalid_spans": row_invalid,
            }
        )

    return pd.DataFrame(rows), pd.DataFrame(spans), pd.DataFrame(issues)

def aggregate_summary(rows_df: pd.DataFrame, spans_df: pd.DataFrame, issues_df: pd.DataFrame) -> pd.DataFrame:
    if rows_df.empty:
        return pd.DataFrame()

    out = []
    for (repo, subset, split), group_rows in rows_df.groupby(["repo", "subset", "split"]):
        group_spans = spans_df[(spans_df.repo == repo) & (spans_df.subset == subset) & (spans_df.split == split)]
        group_issues = issues_df[(issues_df.repo == repo) & (issues_df.subset == subset) & (issues_df.split == split)]

        n_spans = len(group_spans)
        n_valid = int(group_spans["is_valid"].sum()) if n_spans else 0
        n_invalid = n_spans - n_valid

        out.append(
            {
                "repo": repo,
                "subset": subset,
                "split": split,
                "rows": len(group_rows),
                "rows_empty_annotations": int((group_rows["n_annotations"] == 0).sum()),
                "rows_no_valid_spans": int((group_rows["n_valid_spans"] == 0).sum()),
                "rows_any_invalid_spans": int((group_rows["n_invalid_spans"] > 0).sum()),
                "spans_total": n_spans,
                "spans_per_row": (n_spans / len(group_rows)) if len(group_rows) else np.nan,
                "spans_invalid": n_invalid,
                "spans_invalid_rate": (n_invalid / n_spans) if n_spans else np.nan,
                "label_pos(1.0)": int((group_spans["label_scalar"] == 1.0).sum()) if n_spans else 0,
                "label_neg(0.0)": int((group_spans["label_scalar"] == 0.0).sum()) if n_spans else 0,
                "label_ignore(-100)": int((group_spans["label_scalar"] == -100.0).sum()) if n_spans else 0,
                "label_unknown": int(group_spans["label_scalar"].isna().sum()) if n_spans else 0,
                "completion_len_mean": float(group_rows["completion_len"].mean()),
                "completion_len_median": float(group_rows["completion_len"].median()),
                "span_len_mean": float(group_spans["span_len_chars"].mean()) if n_spans else np.nan,
                "span_len_median": float(group_spans["span_len_chars"].median()) if n_spans else np.nan,
                "span_len_p90": float(group_spans["span_len_chars"].quantile(0.9)) if n_spans else np.nan,
                "top_issue": group_issues["issue"].value_counts().index[0] if len(group_issues) else None,
                "top_issue_count": int(group_issues["issue"].value_counts().iloc[0]) if len(group_issues) else 0,
                "label_vocab_size": int(group_spans["label_raw"].nunique(dropna=True)) if n_spans else 0,
            }
        )

    return pd.DataFrame(out).sort_values(["repo", "subset", "split"]).reset_index(drop=True)

def compare_longfact_parsed(our_rows: pd.DataFrame, our_spans: pd.DataFrame, orig_rows: pd.DataFrame, orig_spans: pd.DataFrame) -> pd.DataFrame:
    def _summary(name: str, rows: pd.DataFrame, spans: pd.DataFrame) -> dict:
        known = spans[spans["label_scalar"].isin([0.0, 1.0])]
        return {
            "dataset": name,
            "rows": len(rows),
            "spans": len(spans),
            "mean_completion_len": float(rows["completion_len"].mean()) if len(rows) else np.nan,
            "mean_annotations_per_row": float(rows["n_annotations"].mean()) if len(rows) else np.nan,
            "missing_index_rate": float(spans["index"].isna().mean()) if len(spans) else np.nan,
            "unknown_label_rate": float(spans["label_scalar"].isna().mean()) if len(spans) else np.nan,
            "positive_ratio_known_labels": float((known["label_scalar"] == 1.0).mean()) if len(known) else np.nan,
        }

    return pd.DataFrame([
        _summary("our", our_rows, our_spans),
        _summary("original", orig_rows, orig_spans),
    ])

def split_overlap_stats(rows: pd.DataFrame, subset: str) -> dict:
    subset_rows = rows[rows["subset"] == subset]
    train_hashes = set(subset_rows[subset_rows["split"] == "train"]["prompt_hash"].dropna())
    test_hashes = set(subset_rows[subset_rows["split"] == "test"]["prompt_hash"].dropna())
    overlap = train_hashes.intersection(test_hashes)
    return {
        "subset": subset,
        "train_unique_prompts": len(train_hashes),
        "test_unique_prompts": len(test_hashes),
        "overlap_prompts": len(overlap),
        "overlap_rate_vs_test": (len(overlap) / len(test_hashes)) if len(test_hashes) else np.nan,
    }

In [32]:
# Load and analyze selected subsets/splits
targets = [(OUR_REPO, subset, split) for subset in OUR_SUBSETS for split in ["train", "test"]]
targets += [(ORIG_REPO, subset, split) for subset in ORIG_SUBSETS for split in ["train", "test"]]

rows_parts, spans_parts, issues_parts = [], [], []
for repo, subset, split in targets:
    ds = safe_load(repo, subset, split)
    if ds is None:
        continue
    part_rows, part_spans, part_issues = analyze_longfact_dataset(ds, repo=repo, subset=subset, split=split)
    rows_parts.append(part_rows)
    spans_parts.append(part_spans)
    issues_parts.append(part_issues)

rows_df = pd.concat(rows_parts, ignore_index=True) if rows_parts else pd.DataFrame()
spans_df = pd.concat(spans_parts, ignore_index=True) if spans_parts else pd.DataFrame()
issues_df = pd.concat(issues_parts, ignore_index=True) if issues_parts else pd.DataFrame()

print(f"rows_df: {rows_df.shape} | spans_df: {spans_df.shape} | issues_df: {issues_df.shape}")
summary_df = aggregate_summary(rows_df, spans_df, issues_df)
summary_df

Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | test: 1993 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | test: 1996 rows
Loaded obalcells/longfact-augmented-annotations | Mistral-Small-24B-Instruct-2501 | train: 1534 rows
Loaded obalcells/longfact-augmented-annotations | Mistral-Small-24B-Instruct-2501 | test: 922 rows
Loaded obalcells/longfact-augmented-annotations | Qwen2.5-7B-Instruct | train: 1524 rows
Loaded obalcells/longfact-augmented-annotations | Qwen2.5-7B-Instruct | test: 913 rows
Loaded obalcells/longfact-augmented-annotations | gemma-2-9b-it | train: 1495 rows
Loaded obalcells/longfact-augmented-annotations | gemma-2-9b-it | test: 819 rows
Loaded obalcells/longfact-augmented-annotations | Llama-3.3-70B-Instruct | train: 7959 rows
Loaded obalcells/longfac

,repo,subset,split,rows,rows_empty_annotations,rows_no_valid_spans,rows_any_invalid_spans,spans_total,spans_per_row,spans_invalid,spans_invalid_rate,label_pos(1.0),label_neg(0.0),label_ignore(-100),label_unknown,completion_len_mean,completion_len_median,span_len_mean,span_len_median,span_len_p90,top_issue,top_issue_count,label_vocab_size
0,obalcells/longfact-augmented-annotations,Llama-3.3-70B-Instruct,test,934,0,0,167,14346,15.359743,402,0.028022,3729,10215,402,0,3579.241970,3337.0,21.401506,16.0,42.0,missing_or_nonint_index_and_span_not_found,402,3
1,obalcells/longfact-augmented-annotations,Llama-3.3-70B-Instruct,train,7959,0,9,1184,124108,15.593416,2601,0.020958,31709,89789,2610,0,3890.236839,3735.0,20.724417,16.0,41.0,missing_or_nonint_index_and_span_not_found,2601,3
2,obalcells/longfact-augmented-annotations,Meta-Llama-3.1-8B-Instruct,test,908,0,0,157,12932,14.242291,486,0.037581,4674,7771,487,0,3564.605727,3376.0,23.534565,17.0,47.0,missing_or_nonint_index_and_span_not_found,486,3
3,obalcells/longfact-augmented-annotations,Meta-Llama-3.1-8B-Instruct,train,7919,0,8,1164,114322,14.436419,2636,0.023058,40966,70709,2647,0,3687.190428,3474.0,22.432296,16.0,45.0,missing_or_nonint_index_and_span_not_found,2636,3
4,obalcells/longfact-augmented-annotations,Mistral-Small-24B-Instruct-2501,test,922,0,0,215,12364,13.409978,613,0.049579,2448,9303,613,0,3240.859002,3290.5,20.962795,16.0,41.0,missing_or_nonint_index_and_span_not_found,613,3
5,obalcells/longfact-augmented-annotations,Mistral-Small-24B-Instruct-2501,train,1534,0,3,286,21786,14.202086,738,0.033875,3572,17473,741,0,3567.222295,3559.5,19.871248,15.0,39.0,missing_or_nonint_index_and_span_not_found,738,3
6,obalcells/longfact-augmented-annotations,Qwen2.5-7B-Instruct,test,913,0,0,234,10806,11.835706,689,0.063761,4165,5952,689,0,3357.591457,3395.0,23.895891,17.0,47.0,missing_or_nonint_index_and_span_not_found,689,3
7,obalcells/longfact-augmented-annotations,Qwen2.5-7B-Instruct,train,1524,0,2,317,19258,12.636483,866,0.044968,6447,11945,866,0,3782.169291,3757.0,21.596583,16.0,43.0,missing_or_nonint_index_and_span_not_found,866,3
8,obalcells/longfact-augmented-annotations,gemma-2-9b-it,test,819,0,1,189,8727,10.655678,630,0.072190,2099,5997,631,0,2577.642247,2571.0,22.434857,16.0,44.0,missing_or_nonint_index_and_span_not_found,630,3
9,obalcells/longfact-augmented-annotations,gemma-2-9b-it,train,1495,0,4,318,16356,10.940468,791,0.048361,2887,12678,791,0,2836.871572,2781.0,19.473771,15.0,37.0,missing_or_nonint_index_and_span_not_found,791,3


In [ ]:
# Export summary_df for Google Docs paste
from pathlib import Path
from IPython.display import HTML, display

if "summary_df" not in globals() or summary_df.empty:
    raise ValueError("summary_df is missing or empty. Run Cell 6 first.")

export_dir = Path("exports")
export_dir.mkdir(parents=True, exist_ok=True)

export_df = summary_df.copy()

md_path = export_dir / "summary_df.md"
csv_path = export_dir / "summary_df.csv"
html_path = export_dir / "summary_df.html"

export_df.to_markdown(md_path, index=False)
export_df.to_csv(csv_path, index=False)
export_df.to_html(html_path, index=False)

print(f"Saved: {md_path}")
print(f"Saved: {csv_path}")
print(f"Saved: {html_path}")

print("\nCopy-ready HTML table preview (paste into Google Docs):")
display(HTML(export_df.to_html(index=False)))

print("\nCopy-ready Markdown (optional):")
print(export_df.to_markdown(index=False))

the issue `missing_or_nonint_index_and_span_not_found` corresponds to situation when there is no position to map the annotation onto the text generated by an LLM. This could happen when annotator outputs are paraphrased or hallucinated.

In [38]:
# Compare Apertus vs Llama on key metrics
if summary_df.empty:
    print("No summary available.")
else:
    metrics = [
        "rows", "spans_total", "spans_invalid_rate",
        "rows_no_valid_spans", "label_pos(1.0)", "label_neg(0.0)",
        "label_ignore(-100)", "label_unknown",
        "completion_len_mean", "span_len_median", "span_len_p90",
    ]
    cmp = summary_df[summary_df["repo"] == OUR_REPO][["subset", "split"] + metrics].sort_values(["split", "subset"])
    display(cmp)

    known = spans_df[(spans_df["repo"] == OUR_REPO) & (spans_df["label_scalar"].isin([0.0, 1.0]))]
    pos_ratio = (
        known.assign(is_pos=(known["label_scalar"] == 1.0).astype(float))
        .groupby(["subset", "split"], as_index=False)["is_pos"]
        .mean()
        .rename(columns={"is_pos": "positive_ratio"})
        .sort_values(["split", "subset"])
    )
    display(pos_ratio)

,subset,split,rows,spans_total,spans_invalid_rate,rows_no_valid_spans,label_pos(1.0),label_neg(0.0),label_ignore(-100),label_unknown,completion_len_mean,span_len_median,span_len_p90
10,Apertus_8B_Instruct_2509,test,1993,19143,0.025074,10,4452,14082,609,0,2987.035625,19.0,54.0
12,Meta_Llama_3.1_8B_Instruct,test,1996,23522,0.028824,11,6108,16571,843,0,3915.310621,19.0,50.0
11,Apertus_8B_Instruct_2509,train,17986,172513,0.024682,107,39477,127658,5378,0,3014.388914,19.0,52.0
13,Meta_Llama_3.1_8B_Instruct,train,17959,215298,0.027868,97,54798,152933,7567,0,3917.558049,19.0,50.0


,subset,split,positive_ratio
0,Apertus_8B_Instruct_2509,test,0.240207
2,Meta_Llama_3.1_8B_Instruct,test,0.269324
1,Apertus_8B_Instruct_2509,train,0.236198
3,Meta_Llama_3.1_8B_Instruct,train,0.263793


In [39]:
# Split integrity and leakage checks
if rows_df.empty:
    print("No rows available for leakage checks.")
else:
    leak_df = pd.DataFrame([split_overlap_stats(rows_df, subset) for subset in OUR_SUBSETS])
    display(leak_df)

    t_ap = set(rows_df[(rows_df["subset"] == "Apertus_8B_Instruct_2509") & (rows_df["split"] == "test")]["prompt_hash"].dropna())
    t_ll = set(rows_df[(rows_df["subset"] == "Meta_Llama_3.1_8B_Instruct") & (rows_df["split"] == "test")]["prompt_hash"].dropna())
    overlap = t_ap.intersection(t_ll)
    print("Cross-subset test prompt overlap (Apertus vs Llama):", len(overlap))
    print("Fraction of Apertus test prompts also in Llama test:", (len(overlap) / len(t_ap)) if len(t_ap) else np.nan)

,subset,train_unique_prompts,test_unique_prompts,overlap_prompts,overlap_rate_vs_test
0,Apertus_8B_Instruct_2509,17975,1993,0,0.0
1,Meta_Llama_3.1_8B_Instruct,17948,1996,0,0.0


Cross-subset test prompt overlap (Apertus vs Llama): 1993
Fraction of Apertus test prompts also in Llama test: 1.0


## manual review of the issues

Now I want to manually review the rows that seem to be invalid.

In [39]:
# Two-model comparison utilities (clean refactor)
def get_common_prompts(
    repo_a: str,
    subset_a: str,
    split_a: str,
    repo_b: str,
    subset_b: str,
    split_b: str,
) -> pd.DataFrame:
    ds_a = safe_load(repo_a, subset_a, split_a)
    ds_b = safe_load(repo_b, subset_b, split_b)
    if ds_a is None or ds_b is None:
        return pd.DataFrame()

    def _build_prompt_table(ds, side: str) -> pd.DataFrame:
        records = []
        for row_id, row in enumerate(ds):
            parsed = parse_longfact_row(row)
            prompt = parsed.get("prompt") if isinstance(parsed.get("prompt"), str) else ""
            records.append({
                f"{side}_row_id": row_id,
                f"{side}_prompt": prompt,
                "prompt_hash": _prompt_hash(prompt),
            })
        return pd.DataFrame(records)

    a_tbl = _build_prompt_table(ds_a, "a")
    b_tbl = _build_prompt_table(ds_b, "b")

    common = a_tbl.merge(b_tbl, on="prompt_hash", how="inner")
    common["exact_prompt_match"] = common["a_prompt"] == common["b_prompt"]
    return common

def compare_two_models_on_common_prompts(
    repo_a: str,
    subset_a: str,
    split_a: str,
    repo_b: str,
    subset_b: str,
    split_b: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    common = get_common_prompts(repo_a, subset_a, split_a, repo_b, subset_b, split_b)
    if common.empty:
        return pd.DataFrame(), pd.DataFrame()

    a_rows = rows_df[
        (rows_df["repo"] == repo_a)
        & (rows_df["subset"] == subset_a)
        & (rows_df["split"] == split_a)
    ][["row_id", "prompt_hash", "completion_len", "n_annotations", "n_valid_spans", "n_invalid_spans"]].rename(columns={
        "row_id": "a_row_id",
        "completion_len": "a_completion_len",
        "n_annotations": "a_n_annotations",
        "n_valid_spans": "a_n_valid_spans",
        "n_invalid_spans": "a_n_invalid_spans",
    })

    b_rows = rows_df[
        (rows_df["repo"] == repo_b)
        & (rows_df["subset"] == subset_b)
        & (rows_df["split"] == split_b)
    ][["row_id", "prompt_hash", "completion_len", "n_annotations", "n_valid_spans", "n_invalid_spans"]].rename(columns={
        "row_id": "b_row_id",
        "completion_len": "b_completion_len",
        "n_annotations": "b_n_annotations",
        "n_valid_spans": "b_n_valid_spans",
        "n_invalid_spans": "b_n_invalid_spans",
    })

    detail = common[["prompt_hash", "a_row_id", "b_row_id", "exact_prompt_match"]].merge(
        a_rows, on=["prompt_hash", "a_row_id"], how="left"
    ).merge(
        b_rows, on=["prompt_hash", "b_row_id"], how="left"
    )

    detail["delta_invalid_spans_a_minus_b"] = detail["a_n_invalid_spans"] - detail["b_n_invalid_spans"]
    detail["delta_completion_len_a_minus_b"] = detail["a_completion_len"] - detail["b_completion_len"]

    summary = pd.DataFrame([
        {
            "repo_a": repo_a,
            "subset_a": subset_a,
            "split_a": split_a,
            "repo_b": repo_b,
            "subset_b": subset_b,
            "split_b": split_b,
            "n_common_prompts": len(detail),
            "exact_prompt_match_rate": float(detail["exact_prompt_match"].mean()) if len(detail) else np.nan,
            "avg_invalid_spans_a": float(detail["a_n_invalid_spans"].mean()) if len(detail) else np.nan,
            "avg_invalid_spans_b": float(detail["b_n_invalid_spans"].mean()) if len(detail) else np.nan,
            "avg_delta_invalid_spans_a_minus_b": float(detail["delta_invalid_spans_a_minus_b"].mean()) if len(detail) else np.nan,
            "avg_completion_len_a": float(detail["a_completion_len"].mean()) if len(detail) else np.nan,
            "avg_completion_len_b": float(detail["b_completion_len"].mean()) if len(detail) else np.nan,
        }
    ])

    return summary, detail

def inspect_single_shared_prompt(
    repo_a: str,
    subset_a: str,
    split_a: str,
    repo_b: str,
    subset_b: str,
    split_b: str,
    prompt_idx: int = 0,
) -> tuple[dict, dict, pd.DataFrame]:
    common = get_common_prompts(repo_a, subset_a, split_a, repo_b, subset_b, split_b)
    if common.empty:
        raise ValueError("No common prompts found between the selected datasets.")
    if prompt_idx < 0 or prompt_idx >= len(common):
        raise IndexError(f"prompt_idx must be in [0, {len(common) - 1}] but got {prompt_idx}.")

    selected = common.iloc[prompt_idx]
    a_row_id = int(selected["a_row_id"])
    b_row_id = int(selected["b_row_id"])

    ds_a = safe_load(repo_a, subset_a, split_a)
    ds_b = safe_load(repo_b, subset_b, split_b)
    if ds_a is None or ds_b is None:
        raise RuntimeError("Could not load one or both datasets.")

    parsed_a = parse_longfact_row(ds_a[a_row_id])
    parsed_b = parse_longfact_row(ds_b[b_row_id])

    anns_a = pd.DataFrame(parsed_a["annotations"])
    anns_b = pd.DataFrame(parsed_b["annotations"])
    if not anns_a.empty:
        anns_a = anns_a.reset_index(names="ann_id")
    if not anns_b.empty:
        anns_b = anns_b.reset_index(names="ann_id")

    comparison_meta = pd.DataFrame([
        {
            "prompt_idx": prompt_idx,
            "prompt_hash": selected["prompt_hash"],
            "exact_prompt_match": bool(selected["exact_prompt_match"]),
            "a_row_id": a_row_id,
            "b_row_id": b_row_id,
            "a_completion_len": len(parsed_a["completion"]) if isinstance(parsed_a["completion"], str) else np.nan,
            "b_completion_len": len(parsed_b["completion"]) if isinstance(parsed_b["completion"], str) else np.nan,
            "a_annotations": len(parsed_a["annotations"]),
            "b_annotations": len(parsed_b["annotations"]),
        }
    ])

    print("PROMPT")
    print("-" * 100)
    print(parsed_a["prompt"] if isinstance(parsed_a["prompt"], str) else "")

    print("\nDATASET A GENERATION")
    print("-" * 100)
    print(parsed_a["completion"] if isinstance(parsed_a["completion"], str) else "")

    print("\nDATASET B GENERATION")
    print("-" * 100)
    print(parsed_b["completion"] if isinstance(parsed_b["completion"], str) else "")

    print("\nMETADATA")
    display(comparison_meta)

    print("DATASET A ANNOTATIONS")
    display(anns_a)

    print("DATASET B ANNOTATIONS")
    display(anns_b)

    return parsed_a, parsed_b, comparison_meta

In [40]:
# Example usage: choose any two models/datasets
repo_a = OUR_REPO
subset_a = "Meta_Llama_3.1_8B_Instruct"
split_a = "test"

repo_b = ORIG_REPO
subset_b = "Meta-Llama-3.1-8B-Instruct"
split_b = "test"

common_prompts_df = get_common_prompts(repo_a, subset_a, split_a, repo_b, subset_b, split_b)
print(f"Common prompts: {len(common_prompts_df)}")
display(common_prompts_df[["a_row_id", "b_row_id", "prompt_hash", "exact_prompt_match"]].head(20))

comparison_summary_df, comparison_detail_df = compare_two_models_on_common_prompts(
    repo_a, subset_a, split_a, repo_b, subset_b, split_b,
 )

print("\nTwo-model comparison summary:")
display(comparison_summary_df)

print("Top rows with largest invalid-span delta (a - b):")
display(comparison_detail_df.sort_values("delta_invalid_spans_a_minus_b", ascending=False).head(20))

Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | test: 1996 rows
Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | test: 908 rows
Common prompts: 61


,a_row_id,b_row_id,prompt_hash,exact_prompt_match
0,18,92,13a393aa1d858e6a65579c590795cc9e,True
1,35,338,4b46da6b64c485b631b8bbdd7cdcc3c5,True
2,79,154,036bbf7170d8c817294e7381de2f82f3,True
3,103,38,6740bc79466137b553016f048bc60bce,True
4,109,664,d252ac27f5340457003a103d76810f5f,True
5,116,167,ab7dbdaf0b3b3dcfdac3f3e1f0b8c2a2,True
6,120,136,8c339bafe6aa2bc073d307add29b021c,True
7,126,536,466401cf18f0ddea9ab250cb9918a0e2,True
8,127,718,b492a2acf572feb6a85cc069e7624d7d,True
9,129,901,9c54f99e0e9efd3c35f7dae129d0e5aa,True


Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | test: 1996 rows
Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | test: 908 rows

Two-model comparison summary:


,repo_a,subset_a,split_a,repo_b,subset_b,split_b,n_common_prompts,exact_prompt_match_rate,avg_invalid_spans_a,avg_invalid_spans_b,avg_delta_invalid_spans_a_minus_b,avg_completion_len_a,avg_completion_len_b
0,tkwiecinski/longfact-test-split,Meta_Llama_3.1_8B_Instruct,test,obalcells/longfact-augmented-annotations,Meta-Llama-3.1-8B-Instruct,test,61,1.0,0.327869,0.098361,0.229508,4075.245902,3969.57377


Top rows with largest invalid-span delta (a - b):


,prompt_hash,a_row_id,b_row_id,exact_prompt_match,a_completion_len,a_n_annotations,a_n_valid_spans,a_n_invalid_spans,b_completion_len,b_n_annotations,b_n_valid_spans,b_n_invalid_spans,delta_invalid_spans_a_minus_b,delta_completion_len_a_minus_b
33,1f7fd3931edf177292f21f0eabf68d0e,587,48,True,3037,42,33,9,3877,7,7,0,9,-840
39,f06abb79841d274c77b4cdb87fa99af1,840,854,True,6759,14,11,3,6537,20,19,1,2,222
14,c7e3e01e8466885783df17f490fae957,185,28,True,4011,27,25,2,3579,17,17,0,2,432
1,4b46da6b64c485b631b8bbdd7cdcc3c5,35,338,True,6954,27,26,1,3818,24,24,0,1,3136
45,0a43a4e311de0299f07498a9a5514f6f,1260,19,True,5329,17,16,1,6302,11,11,0,1,-973
26,e94b7e8f5f97ce94c4aa02cf55c48180,474,822,True,5423,22,21,1,4659,14,14,0,1,764
40,791c45f49c21bce0c2d28c4b34e79b91,856,647,True,4823,11,10,1,5986,21,21,0,1,-1163
34,c25ff4dba212a348c0f80c0260f72944,653,150,True,3958,8,8,0,3790,19,19,0,0,168
35,7e2cc591d8b5ff376c5c6baa3f067580,715,110,True,4118,1,1,0,4011,14,14,0,0,107
36,a06c069e77afe43f304f7bdf9845da12,779,724,True,4183,5,5,0,4419,15,15,0,0,-236


In [60]:
# Inspect one shared prompt side-by-side (generation + annotations)
inspect_repo_a = ORIG_REPO
inspect_subset_a = ORIG_SUBSETS[4] # "Meta-Llama-3.1-8B-Instruct"
inspect_split_a = "train"

inspect_repo_b = OUR_REPO
inspect_subset_b = OUR_SUBSETS[1] # "Meta_Llama_3.1_8B_Instruct"
inspect_split_b = "train"

inspect_prompt_idx = 25  # change this index to inspect a different shared prompt

parsed_a, parsed_b, inspect_meta = inspect_single_shared_prompt(
    inspect_repo_a,
    inspect_subset_a,
    inspect_split_a,
    inspect_repo_b,
    inspect_subset_b,
    inspect_split_b,
    prompt_idx=inspect_prompt_idx,
 )

Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | train: 7919 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | train: 7919 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
PROMPT
----------------------------------------------------------------------------------------------------
Describe the hepatic first-pass metabolism of propranolol, including enzyme pathways, metabolite formation, and relevant pharmacokinetic studies with specific bioavailability measurements.

DATASET A GENERATION
----------------------------------------------------------------------------------------------------
Propranolol is a non-selective beta-blocker used to treat various cardiovascular conditions, including hypertension, angina, and arrhythmias. It undergoes extensive hepatic first-pass metabolism, which significan

,prompt_idx,prompt_hash,exact_prompt_match,a_row_id,b_row_id,a_completion_len,b_completion_len,a_annotations,b_annotations
0,25,97e7af96e6cb1498776e3a84fdb9ea65,True,28,5038,2860,3372,14,7


DATASET A ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,CYP1A2,363,Supported,0.0
1,1,CYP2D6,504,Supported,0.0
2,2,CYP2C19,653,Supported,0.0
3,3,4-hydroxypropranolol,2641,Supported,0.0
4,4,naphthoxylactic acid,589,Supported,0.0
5,5,naphthoxyacetic acid,620,Supported,0.0
6,6,25%,2764,Supported,0.0
7,7,Journal of Pharmacology and Experimental Thera...,1543,Supported,0.0
8,8,1975,1598,Insufficient Information,1.0
9,9,22%,1842,Insufficient Information,1.0


DATASET B ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,CYP2D6,328,Supported,0.0
1,1,CYP1A2,562,Supported,0.0
2,2,UGT1A1,691,Not Supported,1.0
3,3,4-Hydroxypropranolol,896,Supported,0.0
4,4,5-Hydroxypropranolol,1089,Supported,0.0
5,5,7-Hydroxypropranolol,1238,Not Supported,1.0
6,6,bioavailability of propranolol after oral admi...,1790,Supported,0.0


In [49]:
# Inspect one shared prompt side-by-side (generation + annotations)
inspect_repo_a = ORIG_REPO
inspect_subset_a = ORIG_SUBSETS[4] # "Meta-Llama-3.1-8B-Instruct"
inspect_split_a = "train"

inspect_repo_b = OUR_REPO
inspect_subset_b = OUR_SUBSETS[1] # "Meta_Llama_3.1_8B_Instruct"
inspect_split_b = "train"

inspect_prompt_idx = 2  # change this index to inspect a different shared prompt

parsed_a, parsed_b, inspect_meta = inspect_single_shared_prompt(
    inspect_repo_a,
    inspect_subset_a,
    inspect_split_a,
    inspect_repo_b,
    inspect_subset_b,
    inspect_split_b,
    prompt_idx=inspect_prompt_idx,
 )

Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | train: 7919 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | train: 7919 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
PROMPT
----------------------------------------------------------------------------------------------------
What constitutes a heart-healthy diet according to current research?

DATASET A GENERATION
----------------------------------------------------------------------------------------------------
According to current research, a heart-healthy diet is characterized by the following key components:

1. **High intake of fruits and vegetables**: Aim for at least 5 servings a day, with a variety of colors to ensure a range of vitamins and minerals.
2. **Whole grains**: Choose whole grains over refined or processed grains, such 

,prompt_idx,prompt_hash,exact_prompt_match,a_row_id,b_row_id,a_completion_len,b_completion_len,a_annotations,b_annotations
0,2,2c659bbe5cc6ba6833ff9b48f51f9740,True,3,7810,2241,2571,6,7


DATASET A ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,5 servings a day,165,Supported,0.0
1,1,"2,300 milligrams of sodium per day",830,Supported,0.0
2,2,25-30 grams of fiber per day,1243,Supported,0.0
3,3,DASH Diet,1718,Supported,0.0
4,4,Dietary Approaches to Stop Hypertension,1742,Supported,0.0
5,5,Mediterranean Diet,1557,Supported,0.0


DATASET B ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,5 servings a day,166,Supported,0.0
1,1,"2,300 milligrams of sodium",837,Supported,0.0
2,2,"1,500 milligrams",898,Supported,0.0
3,3,10% of daily calories,1117,Supported,0.0
4,4,Mediterranean diet,2037,Supported,0.0
5,5,DASH diet,2141,Supported,0.0
6,6,Plant-based diet,2237,Supported,0.0


Prompt 2 in train llama seems identical - the annotations are also very similar

In [ ]:
# Inspect one shared prompt side-by-side (generation + annotations)
inspect_repo_a = ORIG_REPO
inspect_subset_a = ORIG_SUBSETS[4] # "Meta-Llama-3.1-8B-Instruct"
inspect_split_a = "train"

inspect_repo_b = OUR_REPO
inspect_subset_b = OUR_SUBSETS[0] # "Apertus_8B_Instruct_2509"
inspect_split_b = "train"

inspect_prompt_idx = 2  # change this index to inspect a different shared prompt

parsed_a, parsed_b, inspect_meta = inspect_single_shared_prompt(
    inspect_repo_a,
    inspect_subset_a,
    inspect_split_a,
    inspect_repo_b,
    inspect_subset_b,
    inspect_split_b,
    prompt_idx=inspect_prompt_idx,
 )

In [ ]:
# Inspect one shared prompt side-by-side (generation + annotations)
inspect_repo_a = ORIG_REPO
inspect_subset_a = ORIG_SUBSETS[4] # "Meta-Llama-3.1-8B-Instruct"
inspect_split_a = "train"

inspect_repo_b = OUR_REPO
inspect_subset_b = OUR_SUBSETS[0] # "Apertus_8B_Instruct_2509"
inspect_split_b = "train"

inspect_prompt_idx = 2  # change this index to inspect a different shared prompt

parsed_a, parsed_b, inspect_meta = inspect_single_shared_prompt(
    inspect_repo_a,
    inspect_subset_a,
    inspect_split_a,
    inspect_repo_b,
    inspect_subset_b,
    inspect_split_b,
    prompt_idx=inspect_prompt_idx,
 )

Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | train: 7919 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
Loaded obalcells/longfact-augmented-annotations | Meta-Llama-3.1-8B-Instruct | train: 7919 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
PROMPT
----------------------------------------------------------------------------------------------------
What constitutes a heart-healthy diet according to current research?

DATASET A GENERATION
----------------------------------------------------------------------------------------------------
According to current research, a heart-healthy diet is characterized by the following key components:

1. **High intake of fruits and vegetables**: Aim for at least 5 servings a day, with a variety of colors to ensure a range of vitamins and minerals.
2. **Whole grains**: Choose whole grains over refined or processed grains, such as b

,prompt_idx,prompt_hash,exact_prompt_match,a_row_id,b_row_id,a_completion_len,b_completion_len,a_annotations,b_annotations
0,2,2c659bbe5cc6ba6833ff9b48f51f9740,True,3,7121,2241,1617,6,4


DATASET A ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,5 servings a day,165,Supported,0.0
1,1,"2,300 milligrams of sodium per day",830,Supported,0.0
2,2,25-30 grams of fiber per day,1243,Supported,0.0
3,3,DASH Diet,1718,Supported,0.0
4,4,Dietary Approaches to Stop Hypertension,1742,Supported,0.0
5,5,Mediterranean Diet,1557,Supported,0.0


DATASET B ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,5 servings a day,116,Supported,0.0
1,1,"2,300 milligrams of sodium per day",916,Supported,0.0
2,2,up to one drink per day for women,1295,Supported,0.0
3,3,up to two drinks per day for men,1333,Supported,0.0


In [53]:
# Inspect one shared prompt side-by-side (generation + annotations)
inspect_repo_a = OUR_REPO
inspect_subset_a = OUR_SUBSETS[1] # "Meta-Llama-3.1-8B-Instruct"
inspect_split_a = "train"

inspect_repo_b = OUR_REPO
inspect_subset_b = OUR_SUBSETS[0] # "Apertus_8B_Instruct_2509"
inspect_split_b = "train"

inspect_prompt_idx = 5  # change this index to inspect a different shared prompt

parsed_a, parsed_b, inspect_meta = inspect_single_shared_prompt(
    inspect_repo_a,
    inspect_subset_a,
    inspect_split_a,
    inspect_repo_b,
    inspect_subset_b,
    inspect_split_b,
    prompt_idx=inspect_prompt_idx,
 )

Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
PROMPT
----------------------------------------------------------------------------------------------------
How do you establish the existence and uniqueness of solutions to ordinary differential equations?

DATASET A GENERATION
----------------------------------------------------------------------------------------------------
Establishing the existence and uniqueness of solutions to ordinary differential equations (ODEs) is a fundamental problem in mathematics and is crucial for understanding the behavior of physical systems. Here are the general steps to establish the existence and uniqueness of solutions to ODEs:

**Existence Theorem:**

Th

,prompt_idx,prompt_hash,exact_prompt_match,a_row_id,b_row_id,a_completion_len,b_completion_len,a_annotations,b_annotations
0,2,ba8d7ca03268f4d2ee8e06ab77f0d02e,True,2,180,3350,1478,2,6


DATASET A ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,Picard-Lindelof theorem,1850,Supported,0.0
1,1,Cauchy-Lipschitz theorem,2133,Supported,0.0


DATASET B ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,Existence Theorem,138,Insufficient Information,1.0
1,1,Picard-Lindelöf Theorem,157,Supported,0.0
2,2,Lipschitz condition,251,Supported,0.0
3,3,Uniqueness Theorem,467,Insufficient Information,1.0
4,4,Peano Theorem,728,Supported,0.0
5,5,Numerical Methods,959,Supported,0.0


In [58]:
# Inspect one shared prompt side-by-side (generation + annotations)
inspect_repo_a = OUR_REPO
inspect_subset_a = OUR_SUBSETS[1] # "Meta-Llama-3.1-8B-Instruct"
inspect_split_a = "train"

inspect_repo_b = OUR_REPO
inspect_subset_b = OUR_SUBSETS[0] # "Apertus_8B_Instruct_2509"
inspect_split_b = "train"

inspect_prompt_idx = 25  # change this index to inspect a different shared prompt

parsed_a, parsed_b, inspect_meta = inspect_single_shared_prompt(
    inspect_repo_a,
    inspect_subset_a,
    inspect_split_a,
    inspect_repo_b,
    inspect_subset_b,
    inspect_split_b,
    prompt_idx=inspect_prompt_idx,
 )

Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
Loaded tkwiecinski/longfact-test-split | Meta_Llama_3.1_8B_Instruct | train: 17959 rows
Loaded tkwiecinski/longfact-test-split | Apertus_8B_Instruct_2509 | train: 17986 rows
PROMPT
----------------------------------------------------------------------------------------------------
Describe the development timeline and regulatory approval process for nemiralisib in treating COPD exacerbations.

DATASET A GENERATION
----------------------------------------------------------------------------------------------------
I am unable to verify the development timeline and regulatory approval process for nemiralisib in treating COPD exacerbations.

DATASET B GENERATION
----------------------------------------------------------------------------------------------------
Nemiralisib is a medication that is currently under devel

,prompt_idx,prompt_hash,exact_prompt_match,a_row_id,b_row_id,a_completion_len,b_completion_len,a_annotations,b_annotations
0,25,823cb32e4f1b86061e84339286fc2ce8,True,25,258,126,2974,0,3


DATASET A ANNOTATIONS


""


DATASET B ANNOTATIONS


,ann_id,span,index,label,label_scalar
0,0,nemiralisib,180,Supported,0.0
1,1,COPD exacerbations,1022,Supported,0.0
2,2,2024,2711,Insufficient Information,1.0
